# Fine-tune the ScholarCrew writer with LoRA (Kaggle)

Trains a small adapter on top of Qwen3-8B, using the checked examples built by `scholarcrew.finetune.build_dataset`.

**Before you start, in the right-hand panel:**
1. Settings > **Accelerator** > **GPU T4 x2** (choose T4, not P100).
2. Settings > **Internet** > **On** (needs a phone-verified Kaggle account).
3. **Add Input** > **Upload** > create a **private** dataset containing your `train.jsonl`.

Then run the cells from top to bottom.

In [ ]:
# 1. Check the GPU. You should see two Tesla T4 cards.
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Get the code and the libraries
%cd /kaggle/working
!git clone -q https://github.com/omsawant-ds/scholarcrew.git
%cd scholarcrew
!pip install -q -U transformers peft accelerate bitsandbytes

In [ ]:
# 3. Find your uploaded training examples and put them where the script expects them.
import glob, os, shutil

found = glob.glob('/kaggle/input/**/train.jsonl', recursive=True)
assert found, 'train.jsonl not found. Add your private dataset with Add Input > Upload.'
os.makedirs('data/sft', exist_ok=True)
shutil.copy(found[0], 'data/sft/train.jsonl')
print('using', found[0])
!wc -l data/sft/train.jsonl

In [ ]:
# 4. Train. About 30 to 60 minutes on one T4 for 8B with ~100 short examples.
#    Uses one GPU on purpose. The model download (about 16 GB) goes to temporary disk.
#    If the GPU still runs out of memory, change the model to Qwen/Qwen3-4B.
!CUDA_VISIBLE_DEVICES=0 HF_HOME=/kaggle/temp/hf PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True PYTHONPATH=src python -m scholarcrew.finetune.train_lora --model Qwen/Qwen3-8B --out outputs/writer-lora

In [ ]:
# 5. Package the adapter so it appears under Output and can be downloaded.
import os
assert os.path.exists('outputs/writer-lora/adapter_model.safetensors'), 'Training did not finish: there is no adapter to download. Look at the error in cell 4.'
!cd outputs && zip -qr /kaggle/working/writer-lora.zip writer-lora -x 'writer-lora/checkpoints/*'
!ls -lh /kaggle/working/writer-lora.zip

**To download the adapter:** open the right-hand panel > **Data** > **Output** > `/kaggle/working/writer-lora.zip` > the three dots > **Download**.

If you close the notebook first, use **Save Version > Save & Run All** instead, and download the zip from the version's **Output** tab.